In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Adım 1: Yapılandırılmış Çıktılar için Pydantic Modellerini Tanımlama

Bu modeller, ajanların döndüreceği **şemayı** tanımlar. Pydantic ile `response_format` kullanmak şunları sağlar:
- ✅ Tür güvenli veri çıkarımı
- ✅ Otomatik doğrulama
- ✅ Serbest metin yanıtlarından kaynaklanan ayrıştırma hatası olmaz
- ✅ Alanlara göre kolay koşullu yönlendirme


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Adım 2: Otel Rezervasyon Aracını Oluşturun

Bu araç, **availability_agent**'ın odaların müsait olup olmadığını kontrol etmek için çağıracağı araçtır. `@ai_function` dekoratörü ile:
- Bir Python işlevini AI tarafından çağrılabilir bir araca dönüştürürüz
- LLM için otomatik olarak JSON şeması oluştururuz
- Parametre doğrulamasını yönetiriz
- Temsilciler tarafından otomatik çağrılmayı sağlar

Bu demo için:
- **Stockholm, Seattle, Tokyo, Londra, Amsterdam** → Odalar mevcut ✅
- **Diğer tüm şehirler** → Odalar mevcut değil ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Adım 3: Yönlendirme için Koşul Fonksiyonlarını Tanımlama

Bu fonksiyonlar, ajanın yanıtını inceler ve iş akışında hangi yolun izleneceğine karar verir.

**Ana Desen:**
1. Mesajın `AgentExecutorResponse` olup olmadığını kontrol et
2. Yapılandırılmış çıktıyı (Pydantic modeli) ayrıştır
3. Yönlendirmeyi kontrol etmek için `True` veya `False` döndür

İş akışı, hangi yürütücünün sırada çağrılacağını belirlemek için bu koşulları **kenarlarda** değerlendirecektir.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## Adım 4: Özel Gösterim Yürütücüsü Oluşturma

Yürütücüler, dönüşümler veya yan etkiler gerçekleştiren iş akışı bileşenleridir. Sonucu görüntüleyen özel bir yürütücü oluşturmak için `@executor` dekoratörünü kullanıyoruz.

**Önemli Kavramlar:**
- `@executor(id="...")` - Bir fonksiyonu iş akışı yürütücüsü olarak kaydeder
- `WorkflowContext[Never, str]` - Girdi/çıktı için tür ipuçları
- `ctx.yield_output(...)` - Nihai iş akışı sonucunu verir


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## Adım 5: Ortam Değişkenlerini Yükleyin

LLM istemcisini yapılandırın. Bu örnek ile çalışır:
- **GitHub Modelleri** (GitHub belirteci ile Ücretsiz katman)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Adım 6: Yapılandırılmış Çıktılarla AI Ajanları Oluşturma

Her biri bir `AgentExecutor` ile sarılmış **üç uzmanlaşmış ajan** oluşturuyoruz:

1. **availability_agent** - Aracı kullanarak otel uygunluğunu kontrol eder
2. **alternative_agent** - Alternatif şehirler önerir (oda yoksa)
3. **booking_agent** - Rezervasyon yapmayı teşvik eder (oda varsa)

**Temel Özellikler:**
- `tools=[hotel_booking]` - Ajan için aracı sağlar
- `response_format=PydanticModel` - Yapılandırılmış JSON çıktısını zorlar
- `AgentExecutor(..., id="...")` - Ajanı iş akışı kullanımı için sarar


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Adım 7: Koşullu Kenarlarla İş Akışını Oluşturma

Şimdi `WorkflowBuilder` kullanarak koşullu yönlendirme ile grafiği oluşturalım:

**İş Akışı Yapısı:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**Ana Yöntemler:**
- `.set_start_executor(...)` - Giriş noktasını ayarlar
- `.add_edge(from, to, condition=...)` - Koşullu kenar ekler
- `.build()` - İş akışını tamamlar


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Adım 8: Test Vakasını Çalıştır 1 - Mevcutluğu OLMAYAN Şehir (Paris)

Simülasyonumuzda odası olmayan Paris'teki otelleri isteyerek **mevcutluk olmama** yolunu test edelim.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Step 9: Test Durumu 2'yi Çalıştır - Mevcut Olan Şehir (Stockholm)

Şimdi simülasyonumuzda odaları olan Stockholm'deki otelleri talep ederek **mevcudiyet** yolunu test edelim.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## Anahtar Çıkarımlar ve Sonraki Adımlar

### ✅ Öğrendikleriniz:

1. **WorkflowBuilder Deseni**
   - Giriş noktasını tanımlamak için `.set_start_executor()` kullanın
   - Koşullu yönlendirme için `.add_edge(from, to, condition=...)` kullanın
   - İş akışını sonlandırmak için `.build()` çağırın

2. **Koşullu Yönlendirme**
   - Koşul fonksiyonları `AgentExecutorResponse` öğesini inceler
   - Yapılandırılmış çıktıların ayrıştırılması ile yönlendirme kararları verilir
   - Bir kenarı etkinleştirmek için `True`, atlamak için `False` döner

3. **Araç Entegrasyonu**
   - Python fonksiyonlarını AI araçlarına dönüştürmek için `@ai_function` kullanın
   - Ajanlar ihtiyaç duyduklarında araçları otomatik olarak çağırır
   - Araçlar, ajanların ayrıştırabileceği JSON döner

4. **Yapılandırılmış Çıktılar**
   - Tür güvenli veri çıkarımı için Pydantic modelleri kullanın
   - Ajan oluştururken `response_format=MyModel` olarak ayarlayın
   - Yanıtları `Model.model_validate_json()` ile ayrıştırın

5. **Özel Yürütücüler**
   - İş akışı bileşenleri oluşturmak için `@executor(id="...")` kullanın
   - Yürütücüler verileri dönüştürebilir veya yan etkiler oluşturabilir
   - İş akışı sonuçları üretmek için `ctx.yield_output()` kullanın

### 🚀 Gerçek Dünya Uygulamaları:

- **Seyahat Rezervasyonu**: Kullanılabilirliği kontrol edin, alternatifler önerin, seçenekleri karşılaştırın
- **Müşteri Hizmetleri**: Sorun türüne, duyguya, önceliğe göre yönlendirme yapın
- **E-ticaret**: Envanteri kontrol edin, alternatifler önerin, siparişleri işleyin
- **İçerik Moderasyonu**: Toksisite puanlarına, kullanıcı işaretlerine göre yönlendirme yapın
- **Onay İş Akışları**: Tutar, kullanıcı rolü, risk seviyesine göre yönlendirme yapın
- **Çok Aşamalı İşlem**: Veri kalitesi, tamamlanma durumuna göre yönlendirme yapın

### 📚 Sonraki Adımlar:

- Daha karmaşık koşullar ekleyin (birden fazla kriter)
- İş akışı durumu yönetimi ile döngüler uygulayın
- Yeniden kullanılabilir bileşenler için alt iş akışları ekleyin
- Gerçek API'lerle entegrasyon yapın (otel rezervasyonu, envanter sistemleri)
- Hata yönetimi ve yedek yollar ekleyin
- Yerleşik görselleştirme araçları ile iş akışlarını görselleştirin


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
